In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [18]:
df=pd.read_csv('LOAN.csv')
df.head()


,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [19]:
y = df['Status']
X = df.drop('Status', axis=1)


In [20]:
X.replace(['NA','N/A','na','null','NULL','?','--',' '], np.nan, inplace=True)
y.replace(['NA','N/A','na','null','NULL','?','--',' '], np.nan, inplace=True)


In [21]:
mask = y.notnull()
X = X[mask]
y = y[mask]


In [22]:
cat_cols = X.select_dtypes('object').columns
num_cols = X.select_dtypes(exclude='object').columns


In [23]:
for col in num_cols:
    X[col] = X[col].fillna(X[col].median())

for col in cat_cols:
    X[col] = X[col].fillna(X[col].mode().iloc[0])


In [24]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col])


In [25]:
keep_cols = [
    'year','Gender','loan_purpose','income','age','Region',
    'occupancy_type','construction_type',
    'business_or_commercial','submission_of_application'
]

X = X[keep_cols]


In [26]:
X['income_log'] = np.log1p(X['income'])
X['income_age_ratio'] = X['income'] / (X['age'] + 1)

X['age_bucket'] = pd.cut(
    X['age'],
    bins=[0,25,35,45,55,65,120],
    labels=False,
    include_lowest=True
)

X['age_bucket'] = X['age_bucket'].fillna(0)

region_mean = X.groupby('Region')['income'].transform('mean')
region_mean = region_mean.fillna(region_mean.median())

X['income_to_region_mean'] = X['income'] / (region_mean + 1)


In [27]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)

for col in X.columns:
    X[col] = X[col].fillna(X[col].median())

print("Total NaN:", X.isnull().sum().sum())


Total NaN: 0


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [29]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)


In [30]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=2,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_sm, y_train_sm)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [31]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

probs = xgb.predict_proba(X_test)[:, 1]
custom_pred = (probs > 0.6).astype(int)

print(confusion_matrix(y_test, custom_pred))
print(classification_report(y_test, custom_pred))
print("ROC AUC:", roc_auc_score(y_test, probs))


[[10759 11647]
 [ 2475  4853]]
              precision    recall  f1-score   support

           0       0.81      0.48      0.60     22406
           1       0.29      0.66      0.41      7328

    accuracy                           0.53     29734
   macro avg       0.55      0.57      0.51     29734
weighted avg       0.69      0.53      0.56     29734

ROC AUC: 0.6211182321329245
